# ============================================================
# Knowledge Base Chatbot for Bengali Books
# RAG + ChromaDB + BGE-M3 + Gemini + Gradio
# ============================================================

from IPython.display import Markdown, display

display(Markdown("""
# 📚 Bengali Book Knowledge Base Chatbot

### Retrieval-Augmented Generation (RAG)

**Pipeline**

Wikisource  
↓  
Web Crawling  
↓  
Text Cleaning  
↓  
Chunking  
↓  
BGE-M3 Embeddings  
↓  
ChromaDB  
↓  
Semantic Retrieval  
↓  
Gemini LLM  
↓  
Grounded Answer + Chapter Citation

> The chatbot is strictly grounded in the selected book.
"""))

In [ ]:
# ============================================================
# CELL 1 — INSTALL REQUIRED PACKAGES
# ============================================================

!pip -q install -U \
    requests \
    beautifulsoup4 \
    lxml \
    pandas \
    tqdm \
    sentence-transformers \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    langchain-google-genai \
    gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23

In [ ]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES
# ============================================================

import os
import re
import json
import time
import shutil
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote, quote
from tqdm.auto import tqdm

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from langchain_google_genai import ChatGoogleGenerativeAI

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


In [ ]:
# ============================================================
# CELL 3 — PROJECT CONFIGURATION
# ============================================================

BOOK_TITLE = "কপালকুণ্ডলা"

BOOK_AUTHOR = "বঙ্কিমচন্দ্র চট্টোপাধ্যায়"

BOOK_YEAR = "১৮৭০"

BOOK_URL = (
    "https://bn.wikisource.org/wiki/"
    "কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)"
)

WIKISOURCE_DOMAIN = "https://bn.wikisource.org"

WIKISOURCE_API = "https://bn.wikisource.org/w/api.php"

# Vector database
VECTOR_DB_DIR = "/content/chroma_kapalkundala"

COLLECTION_NAME = "kapalkundala_bengali"

# Embedding model
EMBEDDING_MODEL = "BAAI/bge-m3"

# Chunk configuration
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

# Retrieval
TOP_K = 5

# Gemini
LLM_MODEL = "gemini-2.5-flash"

print("📚 Book:", BOOK_TITLE)
print("✍️ Author:", BOOK_AUTHOR)
print("📅 Year:", BOOK_YEAR)
print("🔗 URL:", BOOK_URL)

📚 Book: কপালকুণ্ডলা
✍️ Author: বঙ্কিমচন্দ্র চট্টোপাধ্যায়
📅 Year: ১৮৭০
🔗 URL: https://bn.wikisource.org/wiki/কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)


In [ ]:
# ============================================================
# CELL 4 — HTTP REQUEST HEADERS
# ============================================================

headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(compatible; BengaliBookRAG/1.0; Educational Project)"
    )
}

print("✅ HTTP headers configured.")

✅ HTTP headers configured.


In [ ]:
# ============================================================
# CELL 5 — LOAD GEMINI API KEY
# ============================================================

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "❌ GEMINI_API_KEY not found.\n"
        "Go to Colab Secrets and add GEMINI_API_KEY."
    )

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

print("✅ Gemini API key loaded successfully.")

✅ Gemini API key loaded successfully.


In [ ]:
# ============================================================
# CELL 6 — TEST WIKISOURCE CONNECTION
# ============================================================

response = requests.get(
    BOOK_URL,
    headers=headers,
    timeout=30
)

print("HTTP Status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError(
        f"❌ Wikisource request failed: {response.status_code}"
    )

print("✅ Wikisource connection successful.")

HTTP Status: 200
✅ Wikisource connection successful.


In [ ]:
# ============================================================
# CELL 7 — GET EXACT WIKISOURCE PAGE TITLE
# ============================================================

BOOK_PAGE_TITLE = unquote(
    BOOK_URL.split("/wiki/", 1)[1]
)

print("Exact Wikisource title:")
print(BOOK_PAGE_TITLE)

Exact Wikisource title:
কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)


In [ ]:
# ============================================================
# CELL 8 — DISCOVER ALL BOOK CHAPTER PAGES
# ============================================================

def discover_book_pages(book_url):
    """
    Discover chapter/subpage URLs from the main Wikisource page.
    """

    response = requests.get(
        book_url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    main_title = unquote(
        book_url.split("/wiki/", 1)[1]
    )

    discovered = []

    for link in soup.find_all("a", href=True):

        href = link.get("href", "").strip()

        if not href.startswith("/wiki/"):
            continue

        href = href.split("#")[0]

        page_title = unquote(
            href.split("/wiki/", 1)[1]
        )

        # Only pages belonging to this book
        if page_title.startswith(main_title + "/"):

            full_url = urljoin(
                WIKISOURCE_DOMAIN,
                href
            )

            if full_url not in discovered:
                discovered.append(full_url)

    return discovered


chapter_pages = discover_book_pages(BOOK_URL)

print("📖 Chapter pages discovered:", len(chapter_pages))

for i, url in enumerate(chapter_pages, 1):
    title = unquote(
        url.split("/wiki/", 1)[1]
    )

    print(f"{i:02d}. {title}")

📖 Chapter pages discovered: 32
01. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/প্রথম_পরিচ্ছেদ
02. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/দ্বিতীয়_পরিচ্ছেদ
03. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/তৃতীয়_পরিচ্ছেদ
04. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/চতুর্থ_পরিচ্ছেদ
05. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/পঞ্চম_পরিচ্ছেদ
06. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/ষষ্ঠ_পরিচ্ছেদ
07. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/সপ্তম_পরিচ্ছেদ
08. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/অষ্টম_পরিচ্ছেদ
09. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/নবম_পরিচ্ছেদ
10. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/দ্বিতীয়_খণ্ড/প্রথম_পরিচ্ছেদ
11. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/দ্বিতীয়_খণ্ড/দ্বিতীয়_পরিচ্ছেদ
12. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/দ্বিতীয়_খণ্ড/তৃতীয়_পরিচ্ছেদ
13. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_

In [ ]:
# ============================================================
# CELL 9 — BUILD FINAL BOOK PAGE LIST
# ============================================================

# Main book page + discovered chapter pages
book_pages = [BOOK_URL] + chapter_pages

# Remove duplicates
book_pages = list(dict.fromkeys(book_pages))

print("=" * 70)
print("CRAWLER VERIFICATION")
print("=" * 70)

print("Book:", BOOK_TITLE)
print("Main page:", BOOK_URL)
print("Total pages:", len(book_pages))

if len(book_pages) <= 1:
    raise RuntimeError(
        "❌ Only the main page was found.\n"
        "The chapter links were not detected from Wikisource."
    )

print()
print("✅ CRAWLER SUCCESSFUL")
print(f"✅ {len(book_pages)} pages discovered.")

CRAWLER VERIFICATION
Book: কপালকুণ্ডলা
Main page: https://bn.wikisource.org/wiki/কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)
Total pages: 33

✅ CRAWLER SUCCESSFUL
✅ 33 pages discovered.


In [ ]:
# ============================================================
# CELL 10 — DISPLAY DISCOVERED PAGES
# ============================================================

print("📚 KAPALKUNDALA BOOK PAGES")
print("=" * 70)

for i, url in enumerate(book_pages, 1):

    title = unquote(
        url.split("/wiki/", 1)[1]
    )

    print(f"{i:02d}. {title}")

print()
print("=" * 70)

print(f"TOTAL PAGES: {len(book_pages)}")

assert len(book_pages) > 1

print("✅ Page discovery verification passed.")

📚 KAPALKUNDALA BOOK PAGES
01. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)
02. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/প্রথম_পরিচ্ছেদ
03. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/দ্বিতীয়_পরিচ্ছেদ
04. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/তৃতীয়_পরিচ্ছেদ
05. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/চতুর্থ_পরিচ্ছেদ
06. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/পঞ্চম_পরিচ্ছেদ
07. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/ষষ্ঠ_পরিচ্ছেদ
08. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/সপ্তম_পরিচ্ছেদ
09. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/অষ্টম_পরিচ্ছেদ
10. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/প্রথম_খণ্ড/নবম_পরিচ্ছেদ
11. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/দ্বিতীয়_খণ্ড/প্রথম_পরিচ্ছেদ
12. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/দ্বিতীয়_খণ্ড/দ্বিতীয়_পরিচ্ছেদ
13. কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)/দ্বিতীয়_খণ্ড/

In [ ]:
# ============================================================
# CELL 11 — TEXT CLEANING FUNCTION
# ============================================================

def clean_bengali_text(text):
    """
    Clean extracted Bengali text while preserving
    meaningful paragraph boundaries.
    """

    # Normalize spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces around line breaks
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


print("✅ Text cleaning function ready.")

✅ Text cleaning function ready.


In [ ]:
# ============================================================
# CELL 12 — ROBUST WIKISOURCE TEXT EXTRACTION
# ============================================================

def extract_page_text(url):
    """
    Extract readable text from a Wikisource page.
    """

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Try the standard Wikisource content area
    content = soup.select_one(
        "#mw-content-text"
    )

    if content is None:
        content = soup.select_one(
            ".mw-parser-output"
        )

    if content is None:
        return ""

    # Remove unwanted elements
    remove_selectors = [
        "script",
        "style",
        "table",
        "nav",
        "footer",
        "form",
        ".navbox",
        ".metadata",
        ".mw-editsection",
        ".reference",
        ".references",
        "sup",
        ".catlinks",
        ".printfooter",
        ".portal"
    ]

    for selector in remove_selectors:

        for element in content.select(selector):
            element.decompose()

    # Extract text
    text = content.get_text(
        separator="\n",
        strip=True
    )

    return clean_bengali_text(text)


print("✅ Wikisource extractor ready.")

✅ Wikisource extractor ready.


In [ ]:
# ============================================================
# CELL 13 — TEST TEXT EXTRACTION
# ============================================================

test_url = book_pages[1]

print("Testing:")
print(test_url)
print("=" * 70)

test_text = extract_page_text(test_url)

print("Characters extracted:", len(test_text))

print()
print("TEXT PREVIEW")
print("=" * 70)

print(test_text[:2000])

if len(test_text) < 100:
    raise RuntimeError(
        "❌ Text extraction failed for the test page."
    )

print()
print("✅ Text extraction test passed.")

Testing:
https://bn.wikisource.org/wiki/%E0%A6%95%E0%A6%AA%E0%A6%BE%E0%A6%B2%E0%A6%95%E0%A7%81%E0%A6%A3%E0%A7%8D%E0%A6%A1%E0%A6%B2%E0%A6%BE_(%E0%A6%AC%E0%A6%99%E0%A7%8D%E0%A6%95%E0%A6%BF%E0%A6%AE%E0%A6%9A%E0%A6%A8%E0%A7%8D%E0%A6%A6%E0%A7%8D%E0%A6%B0_%E0%A6%9A%E0%A6%9F%E0%A7%8D%E0%A6%9F%E0%A7%8B%E0%A6%AA%E0%A6%BE%E0%A6%A7%E0%A7%8D%E0%A6%AF%E0%A6%BE%E0%A6%AF%E0%A6%BC,_%E0%A7%A7%E0%A7%AE%E0%A7%AD%E0%A7%A6)/%E0%A6%AA%E0%A7%8D%E0%A6%B0%E0%A6%A5%E0%A6%AE_%E0%A6%96%E0%A6%A3%E0%A7%8D%E0%A6%A1/%E0%A6%AA%E0%A7%8D%E0%A6%B0%E0%A6%A5%E0%A6%AE_%E0%A6%AA%E0%A6%B0%E0%A6%BF%E0%A6%9A%E0%A7%8D%E0%A6%9B%E0%A7%87%E0%A6%A6
Characters extracted: 5450

TEXT PREVIEW
বঙ্কিমচন্দ্র চট্টোপাধ্যায়
কপালকুণ্ডলা
১৮৭০
(
পৃ.
১
-
৬
)
প্রথম খণ্ড — প্রথম পরিচ্ছেদ
প্রথম খণ্ড — দ্বিতীয় পরিচ্ছেদ
►
কপালকুণ্ডলা
বঙ্কিমচন্দ্র চট্টোপাধ্যায়
প্রথম খণ্ড — প্রথম পরিচ্ছেদ
১-৬
​
কপালকুণ্ডলা।
প্রথম খণ্ড।
প্রথম পরিচ্ছেদ।
সাগরসঙ্গমে।
“Floating straight obedient to the stream”.
Comedy of Errors.
সার্দ্ধ দ্বিশত বৎসর পূর্ব্বে এক দিন মাঘ মাস

In [ ]:
# ============================================================
# CELL 14 — DOCUMENT METADATA
# ============================================================

def create_metadata(url):
    """
    Create metadata for each book page.
    """

    page_title = unquote(
        url.split("/wiki/", 1)[1]
    )

    if page_title == BOOK_PAGE_TITLE:

        chapter = "মূল পৃষ্ঠা"
        section = "সূচিপত্র"

    else:

        relative = page_title.replace(
            BOOK_PAGE_TITLE + "/",
            "",
            1
        )

        parts = [
            part for part in relative.split("/")
            if part.strip()
        ]

        if len(parts) >= 2:

            chapter = parts[-2]
            section = parts[-1]

        elif len(parts) == 1:

            chapter = parts[0]
            section = parts[0]

        else:

            chapter = "Unknown"
            section = "Unknown"

    return {
        "book": BOOK_TITLE,
        "author": BOOK_AUTHOR,
        "year": BOOK_YEAR,
        "chapter": chapter,
        "section": section,
        "source_url": url,
        "page_title": page_title
    }


print("✅ Metadata function ready.")

✅ Metadata function ready.


In [ ]:
# ============================================================
# CELL 15 — INGEST ENTIRE BOOK
# ============================================================

documents = []
failed_pages = []

print("📖 STARTING BOOK INGESTION")
print("=" * 70)

for i, url in enumerate(book_pages, 1):

    try:

        text = extract_page_text(url)

        if not text or len(text) < 100:

            failed_pages.append({
                "url": url,
                "reason": "Text too short or empty"
            })

            print(
                f"⚠️ [{i}/{len(book_pages)}] "
                f"Skipped — insufficient text"
            )

            continue

        metadata = create_metadata(url)

        doc = Document(
            page_content=text,
            metadata=metadata
        )

        documents.append(doc)

        print(
            f"✅ [{i}/{len(book_pages)}] "
            f"{metadata['chapter']} | "
            f"{len(text):,} characters"
        )

        time.sleep(0.2)

    except Exception as e:

        failed_pages.append({
            "url": url,
            "reason": str(e)
        })

        print(
            f"❌ [{i}/{len(book_pages)}] "
            f"Failed: {str(e)[:120]}"
        )


print()
print("=" * 70)
print("INGESTION SUMMARY")
print("=" * 70)

print("Total URLs:", len(book_pages))
print("Successful documents:", len(documents))
print("Failed pages:", len(failed_pages))

total_chars = sum(
    len(doc.page_content)
    for doc in documents
)

print(f"Total characters: {total_chars:,}")

if len(documents) == 0:
    raise RuntimeError(
        "❌ No book content was ingested."
    )

print()
print("✅ BOOK INGESTION COMPLETED")

📖 STARTING BOOK INGESTION
✅ [1/33] মূল পৃষ্ঠা | 1,893 characters
✅ [2/33] প্রথম_খণ্ড | 5,450 characters
✅ [3/33] প্রথম_খণ্ড | 4,871 characters
✅ [4/33] প্রথম_খণ্ড | 4,523 characters
✅ [5/33] প্রথম_খণ্ড | 4,226 characters
✅ [6/33] প্রথম_খণ্ড | 5,918 characters
✅ [7/33] প্রথম_খণ্ড | 7,476 characters
✅ [8/33] প্রথম_খণ্ড | 1,326 characters
✅ [9/33] প্রথম_খণ্ড | 10,772 characters
✅ [10/33] প্রথম_খণ্ড | 3,037 characters
✅ [11/33] দ্বিতীয়_খণ্ড | 4,800 characters
✅ [12/33] দ্বিতীয়_খণ্ড | 5,243 characters
✅ [13/33] দ্বিতীয়_খণ্ড | 4,136 characters
✅ [14/33] দ্বিতীয়_খণ্ড | 1,642 characters
✅ [15/33] দ্বিতীয়_খণ্ড | 3,617 characters
✅ [16/33] দ্বিতীয়_খণ্ড | 6,501 characters
✅ [17/33] তৃতীয়_খণ্ড | 7,660 characters
✅ [18/33] তৃতীয়_খণ্ড | 3,863 characters
✅ [19/33] তৃতীয়_খণ্ড | 7,183 characters
✅ [20/33] তৃতীয়_খণ্ড | 3,277 characters
✅ [21/33] তৃতীয়_খণ্ড | 4,659 characters
✅ [22/33] তৃতীয়_খণ্ড | 6,063 characters
✅ [23/33] তৃতীয়_খণ্ড | 2,314 characters
✅ [24/33] চতুর্থ_খণ্ড | 3,512 charact

In [ ]:
# ============================================================
# CELL 16 — INGESTION QUALITY CHECK
# ============================================================

print("🔍 INGESTION QUALITY CHECK")
print("=" * 70)

if not documents:
    raise RuntimeError(
        "❌ No book content was successfully ingested."
    )

total_chars = sum(
    len(doc.page_content)
    for doc in documents
)

print("📚 Book:", BOOK_TITLE)
print("✍️ Author:", BOOK_AUTHOR)
print("📄 Documents:", len(documents))
print(f"📝 Total characters: {total_chars:,}")

average_chars = total_chars / len(documents)

print(
    f"📊 Average characters/document: "
    f"{average_chars:,.0f}"
)

print()
print("METADATA CHECK")
print("-" * 70)

required_fields = [
    "book",
    "author",
    "year",
    "chapter",
    "section",
    "source_url",
    "page_title"
]

for field in required_fields:

    valid = all(
        field in doc.metadata
        for doc in documents
    )

    if valid:
        print(f"✅ {field}")
    else:
        print(f"❌ {field}")


print()
print("=" * 70)
print("FIRST DOCUMENT")
print("=" * 70)

first_doc = documents[0]

print("Chapter:", first_doc.metadata["chapter"])
print("Section:", first_doc.metadata["section"])
print("URL:", first_doc.metadata["source_url"])

print()
print(first_doc.page_content[:1500])

print()
print("=" * 70)
print("✅ INGESTION QUALITY CHECK PASSED")

🔍 INGESTION QUALITY CHECK
📚 Book: কপালকুণ্ডলা
✍️ Author: বঙ্কিমচন্দ্র চট্টোপাধ্যায়
📄 Documents: 33
📝 Total characters: 160,092
📊 Average characters/document: 4,851

METADATA CHECK
----------------------------------------------------------------------
✅ book
✅ author
✅ year
✅ chapter
✅ section
✅ source_url
✅ page_title

FIRST DOCUMENT
Chapter: মূল পৃষ্ঠা
Section: সূচিপত্র
URL: https://bn.wikisource.org/wiki/কপালকুণ্ডলা_(বঙ্কিমচন্দ্র_চট্টোপাধ্যায়,_১৮৭০)

বঙ্কিমচন্দ্র চট্টোপাধ্যায়
কপালকুণ্ডলা
১৮৭০
কপালকুণ্ডলা
বঙ্কিমচন্দ্র চট্টোপাধ্যায়
প্রচ্ছদ-—
​
কপালকুণ্ডলা।
শ্রীবঙ্কিমচন্দ্র চট্টোপাধ্যায় প্রণীত।
দ্বিতীয় সংস্করণ।
ক লি কা তা।
নূতন সংস্কৃত যন্ত্র।
সং বৎ ১৯২৬।
মূল্য এক টাকা।
​
কপালকুণ্ডলা।
শ্রীবঙ্কিমচন্দ্র চট্টোপাধ্যায় প্রণীত।
দ্বিতীয় সংস্করণ।
ক লি কা তা।
নূতন সংস্কৃত যন্ত্র।
সংবৎ ১৯২৬।
​
Printed by Hari mohan Mookerjea. 12, Fakeer
Chand Mitter’s Street Calcutta.
​
মদগ্রজ
শ্রীযুক্ত বাবু সঞ্জীবচন্দ্র চট্টোপাধ্যায়
মহাশয়কে
এই গ্রন্থ
উপহার
প্রদান করিলাম।
পরিচ্ছেদ
(মূল গ্রন্থে নেই)
সূচী

In [ ]:
# ============================================================
# CELL 17 — SAVE INGESTED DOCUMENTS
# ============================================================

documents_data = []

for doc in documents:

    documents_data.append({
        "text": doc.page_content,
        **doc.metadata
    })


json_path = "/content/kapalkundala_documents.json"

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        documents_data,
        f,
        ensure_ascii=False,
        indent=2
    )


csv_path = "/content/kapalkundala_documents.csv"

pd.DataFrame(
    documents_data
).to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Documents saved.")

print(json_path)
print(csv_path)

✅ Documents saved.
/content/kapalkundala_documents.json
/content/kapalkundala_documents.csv


In [ ]:
# ============================================================
# CELL 18 — DOCUMENT DISTRIBUTION
# ============================================================

df_docs = pd.DataFrame(documents_data)

print("📊 DOCUMENT STATISTICS")
print("=" * 70)

print(
    df_docs[
        ["chapter", "section"]
    ].to_string(index=False)
)

print()
print("Total documents:", len(df_docs))
print("Total characters:", df_docs["text"].str.len().sum())

print()
print("✅ Document statistics generated.")

📊 DOCUMENT STATISTICS
      chapter           section
   মূল পৃষ্ঠা          সূচিপত্র
   প্রথম_খণ্ড    প্রথম_পরিচ্ছেদ
   প্রথম_খণ্ড দ্বিতীয়_পরিচ্ছেদ
   প্রথম_খণ্ড   তৃতীয়_পরিচ্ছেদ
   প্রথম_খণ্ড   চতুর্থ_পরিচ্ছেদ
   প্রথম_খণ্ড    পঞ্চম_পরিচ্ছেদ
   প্রথম_খণ্ড     ষষ্ঠ_পরিচ্ছেদ
   প্রথম_খণ্ড    সপ্তম_পরিচ্ছেদ
   প্রথম_খণ্ড    অষ্টম_পরিচ্ছেদ
   প্রথম_খণ্ড      নবম_পরিচ্ছেদ
দ্বিতীয়_খণ্ড    প্রথম_পরিচ্ছেদ
দ্বিতীয়_খণ্ড দ্বিতীয়_পরিচ্ছেদ
দ্বিতীয়_খণ্ড   তৃতীয়_পরিচ্ছেদ
দ্বিতীয়_খণ্ড   চতুর্থ_পরিচ্ছেদ
দ্বিতীয়_খণ্ড    পঞ্চম_পরিচ্ছেদ
দ্বিতীয়_খণ্ড     ষষ্ঠ_পরিচ্ছেদ
  তৃতীয়_খণ্ড    প্রথম_পরিচ্ছেদ
  তৃতীয়_খণ্ড দ্বিতীয়_পরিচ্ছেদ
  তৃতীয়_খণ্ড   তৃতীয়_পরিচ্ছেদ
  তৃতীয়_খণ্ড   চতুর্থ_পরিচ্ছেদ
  তৃতীয়_খণ্ড    পঞ্চম_পরিচ্ছেদ
  তৃতীয়_খণ্ড     ষষ্ঠ_পরিচ্ছেদ
  তৃতীয়_খণ্ড    সপ্তম_পরিচ্ছেদ
  চতুর্থ_খণ্ড    প্রথম_পরিচ্ছেদ
  চতুর্থ_খণ্ড দ্বিতীয়_পরিচ্ছেদ
  চতুর্থ_খণ্ড   তৃতীয়_পরিচ্ছেদ
  চতুর্থ_খণ্ড   চতুর্থ_পরিচ্ছেদ
  চতুর্থ_খণ্ড    পঞ্চম_পরিচ্ছেদ
  চতুর্থ_খণ্ড     ষষ্ঠ_পরিচ্ছেদ
  চতুর্থ_খণ্ড    স

In [ ]:
# ============================================================
# CELL 19 — TEXT CHUNKING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        "। ",
        "॥ ",
        "? ",
        "! ",
        ", ",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(
    documents
)

print("📦 CHUNKING COMPLETED")
print("=" * 70)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))

print()
print("First chunk:")
print("-" * 70)
print(chunks[0].page_content[:1500])

print()
print("Metadata:")
print(chunks[0].metadata)

📦 CHUNKING COMPLETED
Original documents: 33
Total chunks: 228

First chunk:
----------------------------------------------------------------------
বঙ্কিমচন্দ্র চট্টোপাধ্যায়
কপালকুণ্ডলা
১৮৭০
কপালকুণ্ডলা
বঙ্কিমচন্দ্র চট্টোপাধ্যায়
প্রচ্ছদ-—
​
কপালকুণ্ডলা।
শ্রীবঙ্কিমচন্দ্র চট্টোপাধ্যায় প্রণীত।
দ্বিতীয় সংস্করণ।
ক লি কা তা।
নূতন সংস্কৃত যন্ত্র।
সং বৎ ১৯২৬।
মূল্য এক টাকা।
​
কপালকুণ্ডলা।
শ্রীবঙ্কিমচন্দ্র চট্টোপাধ্যায় প্রণীত।
দ্বিতীয় সংস্করণ।
ক লি কা তা।
নূতন সংস্কৃত যন্ত্র।
সংবৎ ১৯২৬।
​
Printed by Hari mohan Mookerjea. 12, Fakeer
Chand Mitter’s Street Calcutta.
​
মদগ্রজ
শ্রীযুক্ত বাবু সঞ্জীবচন্দ্র চট্টোপাধ্যায়
মহাশয়কে
এই গ্রন্থ
উপহার
প্রদান করিলাম।
পরিচ্ছেদ
(মূল গ্রন্থে নেই)
সূচীপত্র
পরিচ্ছেদ
পৃষ্ঠা
প্রথম খণ্ড
প্রথম খণ্ড — প্রথম পরিচ্ছেদ
১
প্রথম খণ্ড — দ্বিতীয় পরিচ্ছেদ
৬
প্রথম খণ্ড — তৃতীয় পরিচ্ছেদ
১১
প্রথম খণ্ড — চতুর্থ পরিচ্ছেদ
১৫
প্রথম খণ্ড — পঞ্চম পরিচ্ছেদ
১৯
প্রথম খণ্ড — ষষ্ঠ পরিচ্ছেদ
২৫
প্রথম খণ্ড — সপ্তম পরিচ্ছেদ
৩২
প্রথম খণ্ড — অষ্টম পরিচ্ছেদ
৩৩
প্রথম খণ্ড — নবম পরিচ্ছেদ
৪৫
দ

In [ ]:
# ============================================================
# CELL 20 — CHUNK QUALITY CHECK
# ============================================================

if not chunks:
    raise RuntimeError(
        "❌ No chunks were created."
    )

chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("🔍 CHUNK QUALITY CHECK")
print("=" * 70)

print("Total chunks:", len(chunks))
print("Minimum length:", min(chunk_lengths))
print("Maximum length:", max(chunk_lengths))
print(
    "Average length:",
    round(sum(chunk_lengths) / len(chunk_lengths), 2)
)

# Metadata verification
metadata_ok = all(
    "source_url" in chunk.metadata
    and "chapter" in chunk.metadata
    and "section" in chunk.metadata
    for chunk in chunks
)

if not metadata_ok:
    raise RuntimeError(
        "❌ Some chunks are missing source metadata."
    )

print()
print("✅ All chunks contain source metadata.")
print("✅ Chunk quality check passed.")

🔍 CHUNK QUALITY CHECK
Total chunks: 228
Minimum length: 1
Maximum length: 997
Average length: 735.17

✅ All chunks contain source metadata.
✅ Chunk quality check passed.


In [ ]:
# ============================================================
# CELL 21 — BGE-M3 EMBEDDING MODEL
# ============================================================

print("🔄 Loading BGE-M3...")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("✅ BGE-M3 loaded successfully.")

🔄 Loading BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ BGE-M3 loaded successfully.


In [ ]:
# ============================================================
# CELL 22 — CREATE CHROMA VECTOR DATABASE (FIXED)
# ============================================================

import os
import shutil
import gc
import time

# ------------------------------------------------------------
# 1. Close/release previous objects
# ------------------------------------------------------------

try:
    del vector_store
except:
    pass

gc.collect()

# ------------------------------------------------------------
# 2. Use a completely new database directory
# ------------------------------------------------------------

VECTOR_DB_DIR = "/content/kapalkundala_chroma_db_v2"

COLLECTION_NAME = "kapalkundala_bengali_v2"

# ------------------------------------------------------------
# 3. Remove old database if it exists
# ------------------------------------------------------------

if os.path.exists(VECTOR_DB_DIR):

    print("🗑️ Removing old Chroma database...")

    shutil.rmtree(
        VECTOR_DB_DIR,
        ignore_errors=True
    )

    time.sleep(2)


# ------------------------------------------------------------
# 4. Create directory manually
# ------------------------------------------------------------

os.makedirs(
    VECTOR_DB_DIR,
    exist_ok=True
)

# Make sure directory is writable
os.chmod(
    VECTOR_DB_DIR,
    0o755
)

print("📁 Database directory:")
print(VECTOR_DB_DIR)

print()
print("🔍 Checking write permission...")

# ------------------------------------------------------------
# 5. Test write permission
# ------------------------------------------------------------

test_file = os.path.join(
    VECTOR_DB_DIR,
    "write_test.txt"
)

try:

    with open(
        test_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write("Chroma write test")

    os.remove(test_file)

    print("✅ Directory is writable.")

except Exception as e:

    raise RuntimeError(
        f"❌ Directory is not writable: {e}"
    )


# ------------------------------------------------------------
# 6. Create Chroma database
# ------------------------------------------------------------

print()
print("🔄 Creating Chroma vector database...")
print(f"📦 Number of chunks: {len(chunks)}")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=VECTOR_DB_DIR
)

print()
print("=" * 70)
print("✅ CHROMA DATABASE CREATED SUCCESSFULLY")
print("=" * 70)

print("Collection:", COLLECTION_NAME)
print("Directory:", VECTOR_DB_DIR)
print("Chunks:", len(chunks))

📁 Database directory:
/content/kapalkundala_chroma_db_v2

🔍 Checking write permission...
✅ Directory is writable.

🔄 Creating Chroma vector database...
📦 Number of chunks: 228

✅ CHROMA DATABASE CREATED SUCCESSFULLY
Collection: kapalkundala_bengali_v2
Directory: /content/kapalkundala_chroma_db_v2
Chunks: 228


In [ ]:
# ============================================================
# CELL 23 — CREATE RETRIEVER
# ============================================================

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": TOP_K
    }
)

print("✅ Retriever created.")
print("Top-K:", TOP_K)

✅ Retriever created.
Top-K: 5


In [ ]:
# ============================================================
# CELL 24 — RETRIEVAL TEST
# ============================================================

test_question = "কপালকুণ্ডলা উপন্যাসের প্রধান চরিত্র কারা?"

retrieved_docs = retriever.invoke(
    test_question
)

print("QUESTION")
print(test_question)

print()
print("=" * 70)

print(
    f"Retrieved documents: {len(retrieved_docs)}"
)

for i, doc in enumerate(
    retrieved_docs,
    1
):

    print()
    print(f"RESULT {i}")
    print("-" * 70)

    print(
        "Chapter:",
        doc.metadata.get("chapter")
    )

    print(
        "Section:",
        doc.metadata.get("section")
    )

    print(
        "Source:",
        doc.metadata.get("source_url")
    )

    print()
    print(
        doc.page_content[:500]
    )

print()
print("✅ Retrieval test completed.")

QUESTION
কপালকুণ্ডলা উপন্যাসের প্রধান চরিত্র কারা?

Retrieved documents: 5

RESULT 1
----------------------------------------------------------------------
Chapter: চতুর্থ_খণ্ড
Section: নবম_পরিচ্ছেদ
Source: https://bn.wikisource.org/wiki/%E0%A6%95%E0%A6%AA%E0%A6%BE%E0%A6%B2%E0%A6%95%E0%A7%81%E0%A6%A3%E0%A7%8D%E0%A6%A1%E0%A6%B2%E0%A6%BE_(%E0%A6%AC%E0%A6%99%E0%A7%8D%E0%A6%95%E0%A6%BF%E0%A6%AE%E0%A6%9A%E0%A6%A8%E0%A7%8D%E0%A6%A6%E0%A7%8D%E0%A6%B0_%E0%A6%9A%E0%A6%9F%E0%A7%8D%E0%A6%9F%E0%A7%8B%E0%A6%AA%E0%A6%BE%E0%A6%A7%E0%A7%8D%E0%A6%AF%E0%A6%BE%E0%A6%AF%E0%A6%BC,_%E0%A7%A7%E0%A7%AE%E0%A7%AD%E0%A7%A6)/%E0%A6%9A%E0%A6%A4%E0%A7%81%E0%A6%B0%E0%A7%8D%E0%A6%A5_%E0%A6%96%E0%A6%A3%E0%A7%8D%E0%A6%A1/%E0%A6%A8%E0%A6%AC%E0%A6%AE_%E0%A6%AA%E0%A6%B0%E0%A6%BF%E0%A6%9A%E0%A7%8D%E0%A6%9B%E0%A7%87%E0%A6%A6

​
সেই জগৎশাসনকর্ত্রী, সুখদুঃখবিধায়িনী কৈবল্যদায়িনী ভৈরবী স্বপ্নে তাঁহার জীবন সমর্পণ আদেশ করিয়াছেন। কেনই বা কপালকুণ্ডলা সে আদেশ পালন না করিবেন?
তুমি আমি প্রাণ ত্যাগ করিতে চাহি না। রাগ করিয়া যাহা বলি

In [ ]:
# ============================================================
# CELL 25 — INITIALIZE GEMINI LLM
# ============================================================

llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    temperature=0,
    max_retries=2
)

print("✅ Gemini LLM initialized.")
print("Model:", LLM_MODEL)

✅ Gemini LLM initialized.
Model: gemini-2.5-flash


In [ ]:
# ============================================================
# CELL 26 — STRICT BENGALI RAG PROMPT
# ============================================================

RAG_PROMPT = """
তুমি একটি Bengali Book Knowledge Base Assistant।

তোমার কাজ হলো শুধুমাত্র নির্বাচিত বইয়ের
প্রদত্ত CONTEXT ব্যবহার করে প্রশ্নের উত্তর দেওয়া।

নিয়ম:

1. শুধুমাত্র CONTEXT থেকে উত্তর দেবে।
2. নিজের সাধারণ জ্ঞান ব্যবহার করবে না।
3. CONTEXT-এ তথ্য না থাকলে অনুমান করবে না।
4. তথ্য না পাওয়া গেলে ঠিক এই বাক্যটি ব্যবহার করবে:

"এই তথ্যটি নির্বাচিত বইয়ের প্রদত্ত অংশে পাওয়া যায়নি।"

5. উত্তর বাংলায় দেবে।
6. সংক্ষিপ্ত কিন্তু পরিষ্কার উত্তর দেবে।
7. উত্তর শেষে সংশ্লিষ্ট Chapter/Section উল্লেখ করবে।
8. সম্ভব হলে source URL উল্লেখ করবে।
9. Context-এর বাইরে কোনো তথ্য যোগ করবে না।

BOOK:
{book}

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

In [ ]:
# ============================================================
# CELL 27 — FORMAT RETRIEVED CONTEXT
# ============================================================

def format_context(docs):

    context_parts = []

    for i, doc in enumerate(docs, 1):

        metadata = doc.metadata

        context_parts.append(
            f"""
--- SOURCE {i} ---

Book: {metadata.get("book")}

Chapter: {metadata.get("chapter")}

Section: {metadata.get("section")}

Source URL: {metadata.get("source_url")}

Text:
{doc.page_content}
"""
        )

    return "\n".join(context_parts)


print("✅ Context formatter ready.")

✅ Context formatter ready.


In [ ]:
# ============================================================
# CELL 28 — COMPLETE RAG PIPELINE
# ============================================================

def answer_question(question):

    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Format context
    context = format_context(docs)

    # Create prompt
    prompt = RAG_PROMPT.format(
        book=BOOK_TITLE,
        context=context,
        question=question
    )

    # Generate answer
    response = llm.invoke(prompt)

    answer = response.content

    # Collect sources
    sources = []

    for doc in docs:

        source = {
            "chapter": doc.metadata.get("chapter"),
            "section": doc.metadata.get("section"),
            "source_url": doc.metadata.get("source_url")
        }

        if source not in sources:
            sources.append(source)

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }


print("✅ RAG pipeline ready.")

✅ RAG pipeline ready.


In [ ]:
# ============================================================
# USE GEMINI 2.5 FLASH-LITE
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

LLM_MODEL = "gemini-2.5-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    google_api_key=GEMINI_API_KEY,
    temperature=0,
    max_retries=2,
    timeout=120,
)

print("✅ Gemini LLM initialized")
print("Model:", LLM_MODEL)

✅ Gemini LLM initialized
Model: gemini-2.5-flash-lite


In [ ]:
# ============================================================
# CELL 29 — FIRST RAG TEST
# ============================================================

question = "কপালকুণ্ডলা উপন্যাসের প্রধান চরিত্র কারা?"

result = answer_question(
    question
)

print("QUESTION")
print("=" * 70)
print(result["question"])

print()
print("ANSWER")
print("=" * 70)
print(result["answer"])

print()
print("SOURCES")
print("=" * 70)

for source in result["sources"]:

    print(
        f"Chapter: {source['chapter']} | "
        f"Section: {source['section']}"
    )

    print(
        f"URL: {source['source_url']}"
    )

print()
print("✅ RAG test completed.")

QUESTION
কপালকুণ্ডলা উপন্যাসের প্রধান চরিত্র কারা?

ANSWER
কপালকুণ্ডলা উপন্যাসের প্রধান চরিত্রদের মধ্যে কপালকুণ্ডলা এবং নবকুমার উল্লেখযোগ্য।

(চতুর্থ_খণ্ড, নবম_পরিচ্ছেদ)
(Source URL: https://bn.wikisource.org/wiki/%E0%A6%95%E0%A6%AA%E0%A6%BE%E0%A6%B2%E0%A6%95%E0%A7%81%E0%A6%A3%E0%A7%8D%E0%A6%A1%E0%A6%B2%E0%A6%BE_(%E0%A6%AC%E0%A6%99%E0%A7%8D%E0%A6%95%E0%A6%BF%E0%A6%AE%E0%A6%9A%E0%A6%A8%E0%A7%8D%E0%A6%A6%E0%A7%8D%E0%A6%B0_%E0%A6%9A%E0%A6%9F%E0%A7%8D%E0%A6%9F%E0%A7%8B%E0%A6%AA%E0%A6%BE%E0%A6%A7%E0%A7%8D%E0%A6%AF%E0%A6%BE%E0%A6%AF%E0%A6%BC,_%E0%A7%A7%E0%A7%AE%E0%A7%AD%E0%A7%A6)/%E0%A6%9A%E0%A6%A4%E0%A7%81%E0%A6%B0%E0%A7%8D%E0%A6%A5_%E0%A6%96%E0%A6%A3%E0%A7%8D%E0%A6%A1/%E0%A6%A8%E0%A6%AC%E0%A6%AE_%E0%A6%AA%E0%A6%B0%E0%A6%BF%E0%A6%9A%E0%A7%8D%E0%A6%9B%E0%A7%87%E0%A6%A6)

SOURCES
Chapter: চতুর্থ_খণ্ড | Section: নবম_পরিচ্ছেদ
URL: https://bn.wikisource.org/wiki/%E0%A6%95%E0%A6%AA%E0%A6%BE%E0%A6%B2%E0%A6%95%E0%A7%81%E0%A6%A3%E0%A7%8D%E0%A6%A1%E0%A6%B2%E0%A6%BE_(%E0%A6%AC%E0%A6%99%E0%A7%8D%E0%A6%

In [ ]:
# ============================================================
# CELL 30 — NO-ANSWER TEST
# ============================================================

no_answer_question = (
    "২০২৬ সালের বাংলাদেশ ক্রিকেট দলের অধিনায়ক কে?"
)

result = answer_question(
    no_answer_question
)

print("QUESTION")
print("=" * 70)
print(result["question"])

print()
print("ANSWER")
print("=" * 70)
print(result["answer"])

QUESTION
২০২৬ সালের বাংলাদেশ ক্রিকেট দলের অধিনায়ক কে?

ANSWER
এই তথ্যটি নির্বাচিত বইয়ের প্রদত্ত অংশে পাওয়া যায়নি।


In [ ]:
# ============================================================
# CELL 31 — EVALUATION QUESTIONS
# ============================================================

test_questions = [
    "কপালকুণ্ডলা উপন্যাসের প্রধান চরিত্র কারা?",
    "কপালকুণ্ডলা কে?",
    "নবকুমার কে?",
    "মতিবিবি কে?",
    "কপালকুণ্ডলার সঙ্গে নবকুমারের কী সম্পর্ক?",
    "উপন্যাসে কাপালিকের ভূমিকা কী?",
    "নবকুমার কীভাবে কপালকুণ্ডলার সঙ্গে পরিচিত হয়?",
    "কপালকুণ্ডলার চরিত্রের প্রধান বৈশিষ্ট্য কী?",
    "উপন্যাসে বনভূমির কী ধরনের বর্ণনা পাওয়া যায়?",
    "২০২৬ সালের বাংলাদেশ ক্রিকেট দলের অধিনায়ক কে?"
]

print("Total evaluation questions:", len(test_questions))

Total evaluation questions: 10


In [ ]:
# ============================================================
# CELL 32 — RUN EVALUATION
# ============================================================

evaluation_results = []

for i, question in enumerate(
    test_questions,
    1
):

    print(
        f"Processing {i}/{len(test_questions)}..."
    )

    result = answer_question(
        question
    )

    evaluation_results.append({
        "id": i,
        "question": question,
        "answer": result["answer"],
        "sources": json.dumps(
            result["sources"],
            ensure_ascii=False
        )
    })


evaluation_df = pd.DataFrame(
    evaluation_results
)

print()
print("✅ Evaluation completed.")

evaluation_df

Processing 1/10...


GoogleRateLimitError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 49.930482326s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-Fr

In [ ]:
# ============================================================
# CELL 33 — SAVE EVALUATION RESULTS
# ============================================================

evaluation_path = (
    "/content/kapalkundala_evaluation.csv"
)

evaluation_df.to_csv(
    evaluation_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Evaluation results saved:")
print(evaluation_path)

NameError: name 'evaluation_df' is not defined

In [ ]:
import gradio as gr

# ------------------------------------------------------------
# CHATBOT FUNCTION
# ------------------------------------------------------------

def chatbot_response(message, history):

    if not message or not message.strip():
        return "⚠️ Please enter a question about the book."

    try:
        result = answer_question(
            message.strip()
        )

        answer = result.get("answer", "").strip()
        sources = result.get("sources", [])

        # Build source information
        source_text = ""

        if sources:
            source_text = "\n\n━━━━━━━━━━━━━━━━━━━━\n📚 **Source Information**\n"

            shown_sources = set()

            for src in sources:
                chapter = src.get("chapter", "Unknown")
                section = src.get("section", "Unknown")

                source_key = f"{chapter}|{section}"

                if source_key in shown_sources:
                    continue

                shown_sources.add(source_key)

                source_text += (
                    f"\n📖 **Chapter:** {chapter}"
                    f"\n📑 **Section:** {section}\n"
                )

        return answer + source_text

    except Exception as e:

        return (
            "⚠️ **Something went wrong while processing your question.**\n\n"
            f"Error: `{str(e)}`"
        )


# ------------------------------------------------------------
# CUSTOM CSS
# ------------------------------------------------------------

custom_css = """

/* ================================
   MAIN PAGE
================================ */

body {
    background: linear-gradient(
        135deg,
        #eef2ff 0%,
        #f8fafc 45%,
        #ecfeff 100%
    ) !important;
}

.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
    padding: 20px !important;
}


/* ================================
   HEADER
================================ */

.book-header {
    background: linear-gradient(
        135deg,
        #4f46e5,
        #7c3aed,
        #0891b2
    );

    border-radius: 24px;

    padding: 30px;

    color: white;

    text-align: center;

    box-shadow:
        0 15px 40px rgba(79,70,229,0.25);

    margin-bottom: 20px;
}

.book-header h1 {
    font-size: 34px;
    margin: 0 0 8px 0;
    font-weight: 800;
}

.book-header p {
    margin: 5px 0;
    font-size: 16px;
    opacity: 0.95;
}


/* ================================
   STATUS CARDS
================================ */

.status-card {
    background: white;

    border-radius: 18px;

    padding: 14px 18px;

    border: 1px solid #e2e8f0;

    box-shadow:
        0 6px 18px rgba(15,23,42,0.06);

    text-align: center;

    margin-bottom: 12px;
}

.status-card strong {
    color: #4f46e5;
}


/* ================================
   CHATBOT
================================ */

#chatbot {

    border-radius: 22px !important;

    border: 1px solid #e2e8f0 !important;

    background: white !important;

    box-shadow:
        0 10px 30px rgba(15,23,42,0.08);

    min-height: 480px;
}


/* ================================
   CHAT MESSAGES
================================ */

.message {
    font-size: 15px !important;
    line-height: 1.75 !important;
}


/* ================================
   INPUT
================================ */

#question-box textarea {

    background: white !important;

    border-radius: 16px !important;

    border: 2px solid #e2e8f0 !important;

    padding: 15px !important;

    font-size: 15px !important;
}

#question-box textarea:focus {

    border-color: #6366f1 !important;

    box-shadow:
        0 0 0 3px rgba(99,102,241,0.12) !important;
}


/* ================================
   BUTTONS
================================ */

#ask-button {

    background: linear-gradient(
        135deg,
        #4f46e5,
        #7c3aed
    ) !important;

    color: white !important;

    border: none !important;

    border-radius: 14px !important;

    font-weight: 700 !important;

    min-height: 48px;
}

#ask-button:hover {

    transform: translateY(-1px);

    box-shadow:
        0 8px 20px rgba(79,70,229,0.25);
}


#clear-button {

    border-radius: 14px !important;

    min-height: 48px;
}


/* ================================
   EXAMPLES
================================ */

.examples-area {

    background: white;

    border-radius: 18px;

    padding: 15px;

    border: 1px solid #e2e8f0;

    margin-top: 15px;
}


/* ================================
   DEVELOPER CARD
================================ */

.developer-card {

    margin-top: 22px;

    padding: 20px;

    border-radius: 20px;

    text-align: center;

    background: linear-gradient(
        135deg,
        #0f172a,
        #1e293b
    );

    color: white;

    box-shadow:
        0 10px 25px rgba(15,23,42,0.15);
}

.developer-card .name {

    font-size: 21px;

    font-weight: 800;

    background: linear-gradient(
        90deg,
        #60a5fa,
        #a78bfa,
        #22d3ee
    );

    -webkit-background-clip: text;

    -webkit-text-fill-color: transparent;
}

.developer-card .role {

    font-size: 13px;

    opacity: 0.75;

    margin-top: 5px;
}


/* ================================
   FOOTER
================================ */

.footer-text {

    text-align: center;

    color: #64748b;

    font-size: 12px;

    margin-top: 12px;
}

"""


# ------------------------------------------------------------
# HEADER
# ------------------------------------------------------------

header_html = """
<div class="book-header">

    <h1>📚 কপালকুণ্ডলা</h1>

    <p>
        Bengali Book Knowledge Base Chatbot
    </p>

    <p>
        Powered by Retrieval-Augmented Generation (RAG)
    </p>

</div>
"""


# ------------------------------------------------------------
# STATUS
# ------------------------------------------------------------

status_html = """
<div class="status-card">

    🟢 <strong>RAG Pipeline Ready</strong>
    &nbsp;&nbsp;•&nbsp;&nbsp;

    📖 Book: <strong>কপালকুণ্ডলা</strong>
    &nbsp;&nbsp;•&nbsp;&nbsp;

    🔎 Retrieval Enabled
    &nbsp;&nbsp;•&nbsp;&nbsp;

    🤖 LLM Connected

</div>
"""


# ------------------------------------------------------------
# DEVELOPER
# ------------------------------------------------------------

developer_html = """
<div class="developer-card">

    <div class="name">
        Md. Ferdaus Hossen
    </div>

    <div class="role">
        AI / ML Engineer • RAG • NLP • LLM Applications
    </div>

</div>

<div class="footer-text">
    Bengali Book Knowledge Base Chatbot • Academic RAG Project
</div>
"""


# ------------------------------------------------------------
# DEMO QUESTIONS
# ------------------------------------------------------------

demo_questions = [

    ["কপালকুণ্ডলা উপন্যাসের লেখক কে?"],

    ["কপালকুণ্ডলার সঙ্গে নবকুমারের প্রথম দেখা কোথায় হয়?"],

    ["কপালকুণ্ডলা কীভাবে নবকুমারের সঙ্গে পরিচিত হয়?"],

    ["নবকুমারের চরিত্র সম্পর্কে বইটিতে কী বলা হয়েছে?"],

    ["কপালকুণ্ডলা ও নবকুমারের সম্পর্ক সম্পর্কে কী জানা যায়?"],

    # No-answer test
    ["কপালকুণ্ডলা উপন্যাসটি ২০২৬ সালে কত কপি বিক্রি হয়েছে?"]

]


# ------------------------------------------------------------
# BUILD GRADIO APP
# ------------------------------------------------------------

with gr.Blocks(
    title="কপালকুণ্ডলা — Bengali Book RAG Chatbot",
    # Removed 'css' and 'theme' from gr.Blocks
) as demo:

    gr.HTML(header_html)

    gr.HTML(status_html)

    # --------------------------------------------------------
    # CHAT
    # --------------------------------------------------------

    chatbot = gr.Chatbot(
        label="💬 Book Assistant",
        elem_id="chatbot",
        height=480,
        # type="messages", # Removed the problematic 'type' argument
        avatar_images=(None, None),
        # show_copy_button=True, # Removed the problematic 'show_copy_button' argument
        # bubble_full_width=False # Removed the problematic 'bubble_full_width' argument
    )

    # --------------------------------------------------------
    # INPUT AREA
    # --------------------------------------------------------

    with gr.Row():

        question_box = gr.Textbox(
            label="Ask a question",
            placeholder=(
                "কপালকুণ্ডলা সম্পর্কে একটি প্রশ্ন লিখুন..."
            ),
            lines=2,
            scale=5,
            elem_id="question-box"
        )

        ask_button = gr.Button(
            "🚀 Ask",
            variant="primary",
            scale=1,
            elem_id="ask-button"
        )

    # --------------------------------------------------------
    # CLEAR
    # --------------------------------------------------------

    with gr.Row():

        clear_button = gr.Button(
            "🧹 Clear Chat",
            variant="secondary",
            elem_id="clear-button"
        )

    # --------------------------------------------------------
    # DEMO QUESTIONS
    # --------------------------------------------------------

    gr.Markdown(
        """
        ### 🎯 Demo Questions

        Select a question below to quickly demonstrate the RAG system.
        """
    )

    gr.Examples(
        examples=demo_questions,
        inputs=question_box,
        label="📖 Suggested Questions",
        examples_per_page=6
    )

    # --------------------------------------------------------
    # PROJECT INFO
    # --------------------------------------------------------

    gr.Markdown(
        """
        ### 🔐 Grounded Answer System

        This chatbot retrieves relevant passages from **কপালকুণ্ডলা**
        and generates answers using the retrieved book context.

        If the requested information is not available in the book,
        the system is designed to respond without inventing an answer.
        """
    )

    gr.HTML(developer_html)


    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    def submit_message(message, history):

        if not message.strip():
            return history, ""

        answer = chatbot_response(
            message,
            history
        )

        history = history or []

        history.append({
            "role": "user",
            "content": message
        })

        history.append({
            "role": "assistant",
            "content": answer
        })

        return history, ""


    ask_button.click(
        submit_message,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    question_box.submit(
        submit_message,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    clear_button.click(
        lambda: ([], ""),
        inputs=None,
        outputs=[chatbot, question_box]
    )


print("✅ Modern Gradio UI created successfully.")


✅ Modern Gradio UI created successfully.


In [ ]:
# ============================================================
# CELL 35 — LAUNCH MODERN GRADIO CHATBOT
# ============================================================

demo.launch(
    share=True,
    debug=False,
    show_error=True,
    inbrowser=True,
    css=custom_css,
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="violet",
        neutral_hue="slate"
    )
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3920253a4565adba71.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# CELL 36 — CHUNKING STRATEGY A
# ============================================================

splitter_a = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        "। ",
        "॥ ",
        "? ",
        "! ",
        " ",
        ""
    ]
)

chunks_a = splitter_a.split_documents(
    documents
)

print("Strategy A")
print("=" * 70)
print("Chunk size: 800")
print("Overlap: 100")
print("Total chunks:", len(chunks_a))

Strategy A
Chunk size: 800
Overlap: 100
Total chunks: 284


In [ ]:
# ============================================================
# CELL 37 — CHUNKING STRATEGY B
# ============================================================

splitter_b = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        "। ",
        "॥ ",
        "? ",
        "! ",
        " ",
        ""
    ]
)

chunks_b = splitter_b.split_documents(
    documents
)

print("Strategy B")
print("=" * 70)
print("Chunk size: 1200")
print("Overlap: 200")
print("Total chunks:", len(chunks_b))

Strategy B
Chunk size: 1200
Overlap: 200
Total chunks: 174


In [ ]:
# ============================================================
# CELL 38 — CHUNKING COMPARISON
# ============================================================

comparison = pd.DataFrame([
    {
        "Strategy": "A",
        "Chunk Size": 800,
        "Overlap": 100,
        "Number of Chunks": len(chunks_a)
    },
    {
        "Strategy": "B",
        "Chunk Size": 1200,
        "Overlap": 200,
        "Number of Chunks": len(chunks_b)
    }
])

comparison

,Strategy,Chunk Size,Overlap,Number of Chunks
0,A,800,100,284
1,B,1200,200,174


In [ ]:
# ============================================================
# CELL 39 — CREATE REQUIREMENTS.TXT
# ============================================================

requirements = """
requests
beautifulsoup4
lxml
pandas
tqdm
sentence-transformers
langchain
langchain-community
langchain-text-splitters
langchain-huggingface
langchain-chroma
chromadb
langchain-google-genai
gradio
"""

with open(
    "/content/requirements.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        requirements.strip()
    )

print("✅ requirements.txt created.")

✅ requirements.txt created.


In [ ]:
# ============================================================
# CELL 41 — FINAL PROJECT CHECK
# ============================================================

print("🔎 FINAL PROJECT CHECK")
print("=" * 70)

checks = {
    "Book URL": bool(BOOK_URL),
    "Book pages": len(book_pages) > 1,
    "Documents": len(documents) > 0,
    "Chunks": len(chunks) > 0,
    "Embeddings": embedding_model is not None,
    "Vector store": vector_store is not None,
    "Retriever": retriever is not None,
    "LLM": llm is not None,
    "Evaluation": len(evaluation_df) == 10,
}

for name, status in checks.items():

    print(
        f"{'✅' if status else '❌'} {name}"
    )

print()
print("=" * 70)

if all(checks.values()):

    print("🎉 PROJECT CHECK PASSED!")
    print("🎉 Bengali Book RAG Chatbot is ready.")

else:

    print(
        "⚠️ Some project checks failed."
    )

🔎 FINAL PROJECT CHECK
✅ Book URL
✅ Book pages
✅ Documents
✅ Chunks
✅ Embeddings
✅ Vector store
✅ Retriever
✅ LLM
✅ Evaluation

🎉 PROJECT CHECK PASSED!
🎉 Bengali Book RAG Chatbot is ready.
